# Product recommendation with a graph neural network

For a retail customer at a weekly cutoff, we rank a set of candidate articles so that the
ones they buy in the next seven days come first.

Along the way, this notebook shows how to:

- sample a two-hop graph neighborhood for each customer, from the customer to their
  recent purchases to the articles in them;
- supply your own candidates with labels that line up with them, stored as a nested
  column of an anchor table that also carries the train / val / test split;
- share encoders between the graph and the candidates, so an article's category gets the
  same ID in both;
- adapt a batch to PyTorch Geometric and train a small heterogeneous GNN;
- measure recall@k on held-out weeks, and save the artifact.

The model and the metric live in [`recommender.py`](recommender.py) next to this
notebook. The contract and the model's input schema are written next to it too
(`contract.json`, `output_schema.json`), so you can read them without running anything.

The data comes from the H&M-style retail generator in `kumo.pytorch.datasets`, at its
smallest size. The [guide](../../docs/04-engines-and-data.md) shows how a contract runs on
the DiskGraph engine.

In [ ]:
import random
import tempfile
from datetime import timedelta
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import torch
from recommender import Recommender, recall_at, rounded, write_json
from torch import nn

from kumo.pytorch import api, datasets

torch.manual_seed(0)
workdir = Path(tempfile.mkdtemp())
data = datasets.hm(workdir / "hm", scale="xs", seed=0)
tables = {
    name: pq.read_table(data.table(name)).to_pylist()
    for name in ("customers", "articles", "transactions")
}
print({name: len(rows) for name, rows in tables.items()})

The DiskGraph engine doesn't support candidate lists yet, so this notebook uses
`InMemoryEngine`, which reads Python rows and is fine at this small size. If you pass this
contract to DiskGraph, `prepare` raises an error naming the candidate set before it reads
any data.

## The schema

The in-memory engine reads Python rows, so every column declares its type explicitly.

In [ ]:
INT64, FLOAT64, STRING = api.DType("int64"), api.DType("float64"), api.DType("string")
TIMESTAMP = api.DType("timestamp", unit="s", timezone="UTC")
DAY = 86_400
ARTICLE_FIELDS = ("product_type_name", "index_group_name", "colour_group_name")

schema = api.DataSchema(
    tables=(
        api.TableSpec(
            "customers",
            "customer_id",
            {"customer_id": INT64, "age": INT64, "club_member_status": STRING},
            time_column=api.NON_TEMPORAL,
        ),
        api.TableSpec(
            "articles",
            "article_id",
            {
                "article_id": INT64,
                **{name: STRING for name in ARTICLE_FIELDS},
                "listed_at": TIMESTAMP,
            },
            time_column="listed_at",
        ),
        api.TableSpec(
            "transactions",
            "transaction_id",
            {
                "transaction_id": INT64,
                "t_dat": TIMESTAMP,
                "customer_id": INT64,
                "article_id": INT64,
                "price": FLOAT64,
            },
            time_column="t_dat",
        ),
    ),
    relations=(
        api.RelationSpec(
            "customer_transactions", "customers", "customer_id", "transactions", "customer_id"
        ),
        api.RelationSpec(
            "article_transactions", "articles", "article_id", "transactions", "article_id"
        ),
    ),
)
declared = {table.name: set(table.columns) for table in schema.tables}
dataset = api.Dataset(
    {
        name: api.InMemoryRows(
            [{k: v for k, v in row.items() if k in declared[name]} for row in rows]
        )
        for name, rows in tables.items()
    }
)

## The anchor table: who, when, which candidates, which were bought

You choose the anchors and the candidates. Here we take every customer who buys in the
week after a cutoff, with the articles they bought plus random articles they didn't, in a
shuffled order, 20 per anchor. The label sits next to each candidate, so labels can't drift
out of alignment with the candidate order. Weeks before October are for training, the next
two weeks for validation, and the last four for testing.

In [ ]:
CANDIDATES = 20
start = min(row["t_dat"] for row in tables["transactions"])
cutoffs = {
    "train": [start + timedelta(days=day) for day in range(119, 295, 7)],
    "val": [start + timedelta(days=day) for day in (301, 308)],
    "test": [start + timedelta(days=day) for day in (315, 322, 329, 336)],
}
article_ids = [row["article_id"] for row in tables["articles"]]
rng = random.Random(0)
anchor_rows = []
for split, times in cutoffs.items():
    for cutoff in times:
        bought: dict[int, set[int]] = {}
        for row in tables["transactions"]:
            if cutoff < row["t_dat"] <= cutoff + timedelta(days=7):
                bought.setdefault(row["customer_id"], set()).add(row["article_id"])
        for customer, positives in sorted(bought.items()):
            positives = sorted(positives)[: CANDIDATES // 2]
            negatives = rng.sample(
                [a for a in article_ids if a not in positives], CANDIDATES - len(positives)
            )
            candidates = [{"article_id": a, "bought": True} for a in positives]
            candidates += [{"article_id": a, "bought": False} for a in negatives]
            rng.shuffle(candidates)
            anchor_rows.append(
                {
                    "sample_id": len(anchor_rows),
                    "customer_id": customer,
                    "cutoff": cutoff,
                    "split": split,
                    "candidates": candidates,
                }
            )
anchors = api.AnchorTable(
    source=api.InMemoryRows(anchor_rows),
    entity_id="customer_id",
    as_of_time="cutoff",
    sample_id="sample_id",
    split="split",
    candidates={
        "articles": api.CandidateColumn(
            column="candidates", entity_id="article_id", targets={"bought": "bought"}
        )
    },
)
print({split: sum(row["split"] == split for row in anchor_rows) for split in cutoffs})

## The contract

- The neighborhood takes each customer's 16 most recent purchases before the cutoff and
  each purchase's article. Graph sampling is point-in-time, so a purchase after the cutoff
  never becomes a node.
- Each candidate is read as of the same cutoff: its attributes, and how often it sold in
  the previous 30 days (a popularity feature).
- Named encoders in `encodings` are shared, so article categories in the graph and in the
  candidate list map to the same IDs.

In [ ]:
sample = api.SampleSpec(
    entity=api.EntitySpec("customers", entity_type="customer"),
    entity_values={"age": api.Field(column="age"), "club": api.Field(column="club_member_status")},
    neighborhoods={
        "recent": api.NeighborhoodSpec(
            layers=(
                api.GraphLayer((api.GraphSample("customer_transactions", 16, strategy="recent"),)),
                api.GraphLayer(
                    (
                        api.GraphSample(
                            "article_transactions", 1, direction="reverse", strategy="recent"
                        ),
                    )
                ),
            )
        )
    },
    candidate_sets={
        "articles": api.CandidateSetSpec(
            entity=api.EntitySpec("articles", entity_type="article"),
            values={
                **{name: api.Field(column=name) for name in ARTICLE_FIELDS},
                "sold_30d": api.Aggregate(
                    api.Rows(("article_transactions",), api.TimeWindow("t_dat", -30 * DAY, 0)),
                    "count",
                ),
            },
            targets={"bought": api.ExternalTarget(dtype=api.DType("bool"))},
        )
    },
)


def shared(name: str) -> api.Encode:
    return api.Encode(name, api.UseEncoding(name))


batch = api.BatchSpec(
    encodings={name: api.Vocabulary() for name in ARTICLE_FIELDS},
    inputs={
        "customer": api.Input(
            api.EntitySource(),
            api.Fields((api.Encode("age", api.ZScore()), api.Encode("club", api.Vocabulary()))),
            api.Separate(),
        ),
        "graph": api.Input(
            api.NeighborhoodSource("recent"),
            api.Graph(
                node_fields={
                    "transactions": api.Fields((api.Encode("price", api.ZScore()),)),
                    "articles": api.Fields(tuple(shared(name) for name in ARTICLE_FIELDS)),
                }
            ),
            None,
        ),
        "candidates": api.Input(
            api.CandidateSource("articles"),
            api.Fields(
                (*(shared(name) for name in ARTICLE_FIELDS), api.Encode("sold_30d", api.ZScore()))
            ),
            api.Padded(),
        ),
    },
    targets={
        "bought": api.CandidateTarget(input="candidates", source="bought", encoding=api.Raw())
    },
    batching=api.Batching(max_samples=64),
)
contract = api.DataLoaderSpec(schema, sample, batch)
engine = api.InMemoryEngine()
issues = api.check(contract=contract, dataset=dataset, engine=engine, anchors=anchors)
print(issues or "the contract checks out")

## One batch, before any model

The graph arrives as node features per table plus an edge index per relation, with each
example's subgraph kept disjoint. Candidates arrive padded, with a validity mask, and the
labels line up with them position by position.

In [ ]:
plan = api.prepare(contract, dataset, engine)
task = plan.materialize(anchors, fit_split="train")
first = next(task.batches(split="train"))
graph = first.inputs["graph"]
print("nodes:", {name: tuple(node["batch"].shape) for name, node in graph["nodes"].items()})
print("edges:", {name: tuple(edge["index"].shape) for name, edge in graph["edges"].items()})
print("candidate IDs:", tuple(first.inputs["candidates"]["index_group_name"]["values"].shape))
print("labels:", tuple(first.targets["bought"]["values"].shape))

## The model

`Recommender` in [`recommender.py`](recommender.py) is ordinary PyTorch Geometric. The
adapter is a few lines: the batch's node features become PyG's `x` dictionary and its edge
indexes become the `edge_index` dictionary. Two rounds of message passing carry article
information through each purchase to the customer; each candidate is embedded from its own
attributes, through the *same* category embeddings, and scored against the customer.

In [ ]:
model = Recommender(task.output_schema, ARTICLE_FIELDS)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
for epoch in range(12):
    model.train()
    losses = []
    for b in task.batches(split="train", epoch=epoch, shuffle=True, seed=0):
        valid = b.inputs["candidates"]["valid"]
        logits = model(b)
        loss = nn.functional.binary_cross_entropy_with_logits(
            logits[valid], b.targets["bought"]["values"][valid].float()
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    print(f"epoch {epoch}: loss {np.mean(losses):.4f}")

## Recall@5 on the test weeks

Recall@5 is the share of an anchor's bought articles that land in its top five. The
baseline ranks candidates by last month's sales alone.

In [ ]:
model.eval()
recall, baseline = [], []
with torch.no_grad():
    for b in task.batches(split="test"):
        valid = b.inputs["candidates"]["valid"]
        bought = b.targets["bought"]["values"].bool() & valid
        recall += recall_at(5, model(b), bought, valid)
        popularity = b.inputs["candidates"]["sold_30d"]["values"].float().nan_to_num()
        baseline += recall_at(5, popularity, bought, valid)
print(f"test recall@5: model {np.mean(recall):.3f}, popularity {np.mean(baseline):.3f}")

## Save the artifact

The artifact, in the temporary directory, holds the contract and every fitted encoder; the
model weights are saved next to it with the fitted state's identity, so they can't be
loaded against a different vocabulary.
[Inference and serving](../../docs/06-inference-and-serving.md) continues from here.

In [ ]:
task.to_artifact().save(workdir / "artifact")
api.save_weights(workdir / "model.pt", model.state_dict(), task.fitted_state)
task.close()
plan.close()
reloaded = api.Artifact.load(workdir / "artifact")
print(sorted(path.name for path in (workdir / "artifact").iterdir()))

## The files next to this notebook

Large files stay in the temporary directory. Two small ones are written next to this
notebook:

- `contract.json`: the contract, byte for byte the file the artifact stores.
- `output_schema.json`: what the model receives, per input and field: dtype, vocabulary
  and size, fitted statistics. Floats are kept to six significant digits so the file is
  the same on every platform.

In [ ]:
Path("contract.json").write_text(reloaded.contract.to_json() + "\n")
write_json("output_schema.json", rounded(reloaded.output_schema))
print({name: len(Path(name).read_bytes()) for name in ("contract.json", "output_schema.json")})

## Check

In [ ]:
assert np.mean(recall) > np.mean(baseline) + 0.05, (np.mean(recall), np.mean(baseline))
assert reloaded.fitted_state == task.fitted_state
assert Path("contract.json").read_text() == (workdir / "artifact" / "contract.json").read_text()
print("checks passed")